목적:
YouTube 영상 URL을 입력받아 STT 서버에 요청하고,
최종 clean transcript와 metadata를 outputs/{video_id}.json 으로 저장한다.

이 노트북은 STT 품질 개선이나 후처리를 고도화하지 않는다.
출력 JSON은 후속 단계의 입력으로 사용한다.

In [1]:
"""
Cell 2. import

- json: STT 서버 응답 파싱 및 결과 저장
- os: 환경변수, 출력 디렉토리 처리
- re: video_id 추출, 노이즈 제거
- shutil: curl 존재 확인
- subprocess: curl 요청
- tempfile: 임시 다운로드 디렉토리
- time: 소요 시간 로그
- datetime: JSON created_at 생성
- pathlib.Path: 경로 처리
- yt_dlp: YouTube audio download
"""

import json
import os
import re
import shutil
import subprocess
import tempfile
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import yt_dlp

In [ ]:
"""
Cell 3. 설정값

여기에 둘 것:
BASE_URL
MODEL
LANGUAGE
RESPONSE_FORMAT
TIMESTAMP_GRANULARITIES
OUTPUT_DIR
YOUTUBE_URL

구성 원칙:
- STT 서버 주소는 환경변수 우선
- 모델명은 qwen3-asr 기본값
- 언어는 ko 기본값
- response_format은 verbose_json 유지
- 출력 디렉토리는 /home/kjh/workspace/ExtracTube/notebooks/outputs
- youtube_url은 여기서만 바꾼다
"""

BASE_URL = os.environ.get("STT_BASE_URL", "http://localhost:8080").rstrip("/")
MODEL = os.environ.get("STT_MODEL", "qwen3-asr")
LANGUAGE = os.environ.get("DEFAULT_LANGUAGE", "ko")
RESPONSE_FORMAT = os.environ.get("STT_RESPONSE_FORMAT", "verbose_json")
TIMESTAMP_GRANULARITIES = [
    x.strip()
    for x in os.environ.get("STT_TIMESTAMP_GRANULARITIES", "segment").split(",")
    if x.strip()
]
OUTPUT_DIR = "/home/kjh/workspace/ExtracTube/notebooks/outputs"
YOUTUBE_URL = "https://www.youtube.com/watch?v=4QV3giY4wgA"

# https://www.youtube.com/watch?v=

In [3]:
"""
Cell 4. 노이즈 패턴 정의

역할:
Qwen3-ASR 출력에 섞일 수 있는 language, Korean asr text 계열 문자열 제거
여기는 실험하지 말고 현재 동작하는 패턴 유지.
"""

NOISE_PATTERNS = [
    re.compile(r"\blanguage\s+[A-Za-z][A-Za-z_-]*\s*<\s*asr[\s_-]*text\s*>", re.IGNORECASE),
    re.compile(r"language\s*(?:Korean\s*asr\s*text|Koreanasrtext)", re.IGNORECASE),
    re.compile(r"Korean\s*asr\s*text", re.IGNORECASE),
    re.compile(r"Koreanasrtext", re.IGNORECASE),
    re.compile(
        r"(?:(?<=^)|(?<=[\s\]\)])|(?<=[\uac00-\ud7af0-9]))language(?=$|[\s\uac00-\ud7af0-9])",
        re.IGNORECASE,
    ),
]

In [4]:
"""
Cell 5. 유틸 함수

포함 함수:

ensure_command(name)
extract_video_id(youtube_url)

역할:

ensure_command:
- curl 없으면 바로 실패

extract_video_id:
- YouTube URL에서 video_id 추출
- outputs/{video_id}.json 파일명 생성에 사용
- youtu.be/VIDEO_ID, youtube.com/watch?v=VIDEO_ID 형식 지원
"""


def ensure_command(name: str):
    if not shutil.which(name):
        raise RuntimeError(f"Required command not found in PATH: {name}")


def extract_video_id(youtube_url):
    patterns = [
        r"youtu\.be/([A-Za-z0-9_-]+)",
        r"[?&]v=([A-Za-z0-9_-]+)",
    ]
    for pattern in patterns:
        match = re.search(pattern, youtube_url)
        if match:
            return match.group(1)
    return "unknown_video"

In [5]:
"""
Cell 6. STT 텍스트 정리 함수

포함 함수:
clean_asr_text(text)
build_clean_transcript(payload)

역할:
clean_asr_text:
- 노이즈 패턴 제거
- 중복 공백 정리
- 문장부호 앞 공백 제거

build_clean_transcript:
- verbose_json의 segments를 paragraph 단위 clean text로 병합
- segment 간 공백이 2.5초 이상이면 문단 분리
- segments가 없으면 payload["text"] 기준으로 fallback
"""


def clean_asr_text(text):
    cleaned = text
    for pattern in NOISE_PATTERNS:
        cleaned = pattern.sub("", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned)
    cleaned = re.sub(r"\s+([,.;:!?])", r"\1", cleaned)
    return cleaned.strip()


def build_clean_transcript(payload):
    segments = payload.get("segments")
    if not segments:
        return clean_asr_text(str(payload.get("text", "")))

    paragraphs = []
    current = []
    previous_end = None

    start_t = time.time()

    for segment in segments:
        text = clean_asr_text(str(segment.get("text", "")))
        if not text:
            continue

        start = segment.get("start")
        if current and previous_end is not None and start is not None and float(start) - previous_end >= 2.5:
            paragraphs.append(" ".join(current))
            current = []

        current.append(text)
        if segment.get("end") is not None:
            previous_end = float(segment["end"])

    if current:
        paragraphs.append(" ".join(current))

    elapsed = time.time() - start_t
    if elapsed > 3.0:
        print(f"[TIME] build_clean_transcript took {elapsed:.2f} seconds")
    return "\n\n".join(paragraphs)

In [6]:
"""
Cell 7. 오디오 다운로드 함수

포함 함수:
download_audio(youtube_url, target_dir)

역할:
- yt-dlp로 YouTube audio 다운로드
- tempfile 내부에 audio 파일 생성
- 다운로드 소요 시간 출력
- 다운로드 실패 시 RuntimeError
- yt-dlp metadata를 후속 JSON 저장 단계로 전달
"""


def download_audio(youtube_url, target_dir):
    ydl_opts = {
        "outtmpl": str(target_dir / "audio.%(ext)s"),
        "format": "bestaudio/best",
        "noplaylist": True,
        "quiet": True,
        "continuedl": False,
        "nooverwrites": True,
    }
    print("[INFO] Downloading full audio with yt-dlp...")
    start_t = time.time()
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=True)
        audio_path = Path(ydl.prepare_filename(info))

    elapsed = time.time() - start_t
    print(f"[TIME] Audio download took {elapsed:.2f} seconds")

    if not audio_path.is_file():
        raise RuntimeError(f"Downloaded file not found: {audio_path}")
    print(f"[INFO] downloaded_audio={audio_path}")
    return audio_path, info

In [7]:
"""
Cell 8. STT 서버 요청 함수

포함 함수:
request_transcription(audio_path)

역할:
- curl로 /v1/audio/transcriptions 호출
- file, model, language, response_format 전달
- timestamp_granularities[] 전달
- timeout은 7200 유지
- 실패 시 stderr/stdout 기반 RuntimeError
- 성공 시 JSON payload와 요청 소요 시간 반환
"""


def request_transcription(audio_path):
    ensure_command("curl")
    form_args = [
        "-F", f"file=@{audio_path}",
        "-F", f"model={MODEL}",
        "-F", f"language={LANGUAGE}",
        "-F", f"response_format={RESPONSE_FORMAT}",
    ]
    for granularity in TIMESTAMP_GRANULARITIES:
        form_args.extend(["-F", f"timestamp_granularities[]={granularity}"])

    print("[INFO] Requesting STT transcription from server...")
    start_t = time.time()
    proc = subprocess.run(
        [
            "curl",
            "-sS",
            "--fail-with-body",
            "-X", "POST",
            *form_args,
            f"{BASE_URL}/v1/audio/transcriptions",
        ],
        capture_output=True,
        text=True,
        check=False,
        timeout=7200,
    )

    elapsed = time.time() - start_t
    print(f"[TIME] STT transcription request took {elapsed:.2f} seconds")

    if proc.returncode != 0:
        message = proc.stderr.strip() or proc.stdout.strip()
        raise RuntimeError(f"STT request failed: {message}")

    payload = json.loads(proc.stdout or "{}")
    if not isinstance(payload, dict):
        raise RuntimeError(f"Unexpected transcription response type: {type(payload).__name__}")
    return payload, elapsed

In [8]:
"""
Cell 9. 출력 저장 함수

포함 함수:
save_stt_result(clean_text, payload, youtube_url, video_info, stt_elapsed_sec, output_dir)

역할:
- video_id 추출
- output_dir 생성
- clean transcript와 metadata를 outputs/{video_id}.json 으로 저장
- 저장 경로 반환

중요한 점:
clean transcript 내용은 transcript 필드에 그대로 저장
raw STT payload 전체는 저장하지 않음
segments는 stats.segment_count 계산에만 사용
"""


def save_stt_result(clean_text, payload, youtube_url, video_info, stt_elapsed_sec, output_dir):
    video_id = extract_video_id(youtube_url)
    os.makedirs(output_dir, exist_ok=True)

    upload_date = video_info.get("upload_date")
    if upload_date:
        upload_date = str(upload_date)
        if re.fullmatch(r"\d{8}", upload_date):
            upload_date = f"{upload_date[:4]}-{upload_date[4:6]}-{upload_date[6:8]}"

    created_at = datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="seconds")
    result = {
        "created_at": created_at,
        "source": {
            "video_id": video_id,
            "url": youtube_url,
            "title": video_info.get("title"),
            "channel": video_info.get("uploader"),
            "upload_date": upload_date,
            "duration_sec": video_info.get("duration"),
            "language": video_info.get("language") or LANGUAGE,
        },
        "stt": {
            "model": MODEL,
            "elapsed_sec": round(float(stt_elapsed_sec), 2),
        },
        "stats": {
            "transcript_char_count": len(clean_text),
            "segment_count": len(payload.get("segments", [])),
        },
        "transcript": clean_text,
    }

    output_path = Path(output_dir) / f"{video_id}.json"
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    return output_path

In [9]:
"""
Cell 10. 실행 셀

역할:
1. 임시 디렉토리 생성
2. YouTube audio 다운로드 및 metadata 확보
3. STT 서버 요청 및 소요 시간 확보
4. clean transcript 생성
5. outputs/{video_id}.json 저장
6. 최종 저장 경로 출력

이 셀이 실제 실행 진입점.

"""

output_path = None
clean_text = ""

with tempfile.TemporaryDirectory(prefix="yt-qwen-asr-") as tmp_dir:
    work_dir = Path(tmp_dir)
    try:
        audio_path, video_info = download_audio(YOUTUBE_URL, work_dir)
        result, stt_elapsed = request_transcription(audio_path)
        clean_text = build_clean_transcript(result)
        output_path = save_stt_result(
            clean_text,
            result,
            YOUTUBE_URL,
            video_info,
            stt_elapsed,
            OUTPUT_DIR,
        )
        print(f"[SUCCESS] STT result saved to {output_path}")
    except Exception as e:
        print(f"[FAIL] {type(e).__name__}: {e}")

[INFO] Downloading full audio with yt-dlp...


[TIME] Audio download took 6.84 seconds                    
[INFO] downloaded_audio=/tmp/yt-qwen-asr-j7kzm35j/audio.webm
[INFO] Requesting STT transcription from server...
[TIME] STT transcription request took 27.10 seconds
[SUCCESS] STT result saved to /home/kjh/workspace/ExtracTube/notebooks/outputs/0GSCyMEQNZQ.json


In [10]:
"""
Cell 11. 결과 확인 셀

역할:

- 저장된 JSON 파일 열기
- 영상 metadata와 전체 글자 수 출력
- transcript 앞부분 1500자만 출력

목적:
STT가 정상적으로 되었는지 빠르게 육안 확인
전체 transcript를 노트북에 매번 다 출력할 필요는 없음. 너무 길어서 가독성만 떨어짐.
"""

if output_path is None:
    output_path = Path(OUTPUT_DIR) / f"{extract_video_id(YOUTUBE_URL)}.json"

with Path(output_path).open("r", encoding="utf-8") as f:
    data = json.load(f)

preview_length = 1500
source = data.get("source", {})
stats = data.get("stats", {})
transcript = data.get("transcript", "")

print(f"[INFO] File path: {output_path}")
print(f"[INFO] Video title: {source.get('title')}")
print(f"[INFO] Channel: {source.get('channel')}")
print(f"[INFO] Duration: {source.get('duration_sec')}")
print(f"[INFO] Total characters: {stats.get('transcript_char_count', len(transcript)):,}")
print("\n======= Transcript Preview =======\n")
print(transcript[:preview_length])
if len(transcript) > preview_length:
    print("\n... [truncated]")

[INFO] File path: /home/kjh/workspace/ExtracTube/notebooks/outputs/0GSCyMEQNZQ.json
[INFO] Video title: 요즘 부자들이 방구석에서 돈 쓸어담는 방법  “이것만 알면 수입이 10배 늘어납니다”  | 📻 들을 수록 소싱 잘되는 앤더슨 라디오
[INFO] Channel: 백만장자 앤더슨
[INFO] Duration: 1468
[INFO] Total characters: 11,117

======= Transcript Preview =======

안녕하세요, 엔더슨입니다. 저희 지옥캠프 수강생분들이 이런 이야기를 많이 하십니다. 엔더슨님 유튜브에 너무 많은 걸 알려주시는 것 같다. 경쟁자가 더 생기니 이제 그만 알려주시면 안 되냐?와 같이 이렇게 유튜브에서 알려드리는 부분에 있어서 그만 해달라는 이야기가 많은데요. 오늘은 제가 6개월 만에 월 매출 2억을 만들었던 방법을 왜 굳이 유튜브에서 알려드리는지에 대해서 이야기해보고자 합니다. 저도 처음엔 다른 분들의 유튜브나 강의를 보면서 왜 이런 돈 버는 방법을 알려주는 거지 혼자만 알면 좋은 건데 사기꾼 아닌가 생각했던 때가 있었어요. 부자들이 이런 방법을 알려주는 이유에 대해서 알게 됐을 땐 머리를 한 대 맞은 것처럼 멍하게 됐고 그 후로 세상을 보는 시야가 달라졌습니다. 그래서 여러분들께 정보나 노하우를 알려드리는 것도 중요하지만 이런 마인드를 가지고 셀러를 임하게 되실 때는. 단기적으로 더 큰 도움이 될 것 같아서 준비해봤습니다. 지난번 첫 영상에서도 댓글 많이 달아주셔서 정말 기분이 좋아한데요. 오늘 영상도 출근하실 때나 상스페이즈 만들 때, 소싱하실 때, 주무시기 전에 틀어놓고 흘러가듯이 들어와주시면 좋을 것 같습니다. 일단은 저도 처음에 사업을 시작을 할 때는 월에 단돈 한 50만 원, 100만 원만이라도 뭔가 나의 힘으로 돈을 만들어 보고 싶다. 이런 이제. 가 컸습니다. 그렇게 해서 뭔가 처음 시작이 됐었는데 이게 막상 단돈 